# 从理论到实践：基于 Transformer Encoder 的新闻分类模型 


1. **🧠 训练过程的深度回放**：机器在跑那 5 步代码时，内部到底发生了什么化学反应？
2. **🚧 基础架构的致命局限性剖析**：为什么把层数 `num_layers` 从 2 改成 100，模型就会当场暴毙？
3. **💡 通往大模型架构的伏笔**：引出 DeepNet 和优化策略的必要性。

## 模块一：准备工具与数据读取
**⚙️ 协同原理**：这是整个流水线的起点。我们会从电脑硬盘里把零散的 CSV 表格文件读出来，把里面的“文字”提取出来，装进 Python 自带的“列表”（大箱子）里，为后面做分词做准备。

In [1]:
# ======= 1. 导入必要的 Python 工具包 (库) =======
# 🐍 语法小课堂：import 表示引入别人写好的代码包。as 是给它起个简短的别名，方便后面写代码。
import os            # os 用于操作文件路径和文件夹
import pandas as pd  # pandas 专门用来处理表格，我们叫它 pd
import torch         # 深度学习老大哥 PyTorch
import torch.nn as nn  # nn 是 Neural Network(神经网络) 的缩写，里面全是网络层
import torch.optim as optim  # optim 是优化器模块，负责做梯度下降
# 🐍 语法小课堂：from ... import ... 表示只从一个大包里拿出我们需要的小工具
from torch.utils.data import Dataset, DataLoader 

# ======= 2. 读取新闻数据集 =======
data_dir = 'news/news-articles-classification-dataset-for-nlp-and-ml' # 字符串变量，存放路径

# 🐍 语法小课堂：列表推导式 [结果 for 变量 in 集合 if 条件]
# 这行代码的意思是：把 data_dir 文件夹里所有的文件名 (f) 拿出来，如果名字是以 '.csv' 结尾的，就留下装进 csv_files 列表(箱子)里。
csv_files = [f for f in os.listdir(data_dir) if f.endswith('.csv')]


最终效果：
csv_files 就是一个列表，比如：
['business_data.csv', 'sports_data.csv', 'tech_data.csv']

In [2]:
# 🐍 语法小课堂：[] 代表列表(List)；{} 代表字典(Dict)。
all_data = []      # 空列表，一会儿用来装所有的 (新闻文字, 类别编号)
labels_map = {}    # 空字典，一会儿用来记录 "business"->0 这样的对应关系

for file in csv_files:
    file_path = os.path.join(data_dir, file)
    df = pd.read_csv(file_path)
    
    # 寻找装有文字的那一列的名字，在我们的数据集中是 'content' 列
    text_col = 'content' if 'content' in df.columns else df.columns[0]
    # 寻找装有类别的那一列的名字，在我们的数据集中是 'category' 列
    label_col = 'category' if 'category' in df.columns else None
    
    for idx, row in df.iterrows():
        # 获取文本和类别
        text = str(row[text_col])
        # 如果有category列就用，没有就回退到使用文件名
        label_name = str(row[label_col]) if label_col else file.replace('_data.csv', '')
        
        # 给遇到的新类别分配一个唯一的编号
        if label_name not in labels_map:
            labels_map[label_name] = len(labels_map)
            
        # 跳过纯空白数据
        if text.strip() and text != 'nan':
            all_data.append((text, labels_map[label_name]))

# 🐍 语法小课堂：f"..." 是格式化字符串，花括号 {} 里的变量会自动填进去
print(f"总共读取了 {len(all_data)} 条新闻数据。")
print(f"类别映射 (字典长这样): {labels_map}")


总共读取了 10000 条新闻数据。
类别映射 (字典长这样): {'business': 0, 'education': 1, 'entertainment': 2, 'sports': 3, 'technology': 4}


## 模块二：文本数字化（编字典）与建立数据流水线
**⚙️ 协同原理**：模块一我们得到了 `all_data` 列表，但全是一串串的中英文。神经网络算不了英文字母。所以这一步，我们要遍历 `all_data`，统计出现最多的词，建立一个“单词->数字编号”的密码本（词表）。
建好密码本后，我们用面向对象编程（Class类）造一个名叫 `NewsDataset` 的“仓库管理员”，专门负责把 `all_data` 里的文字翻译成数字密码并打包成张量，最后交给 `DataLoader` 这个“搬运工”。
这样，后面的模型只要找搬运工要数据，就能直接拿到完全是数字的张量矩阵了！

In [3]:
from collections import Counter

MAX_LEN = 128     # 规定每篇新闻一律截断或补齐到 128 个单词
VOCAB_SIZE = 10000 # 词表最高容量：1万个常见词

# 🐍 语法小课堂：def 用于定义一个函数(相当于一个加工厂)，接收输入 text，加工后用 return 返回结果
def simple_tokenize(text):
    # .lower() 变小写，.split() 默认按空格把句子切成一个个单词的列表
    # 🐍 语法小课堂：切片 [:MAX_LEN] 表示从第 0 个单词一直取到第 MAX_LEN(128) 个单词，后面的直接扔掉
    return str(text).lower().split()[:MAX_LEN]

word_counts = Counter() # Counter 是一种特殊的字典，专门用来数数
for text, _ in all_data: # 遍历刚才收集的所有新闻，下划线 _ 表示我们不关心第二个元素(类别标签)
    word_counts.update(simple_tokenize(text)) # 把切碎的单词扔进去统计词频

# 手动初始化密码本(词表)，预留两个特殊号码（这里的0用作填充符，这里的1用作生僻词（不在词表））
vocab = {'<PAD>': 0, '<UNK>': 1}

# .most_common 取出最常出现的前 9998 个词，返回的是 [(词1, 次数), (词2, 次数)...]
for word, _ in word_counts.most_common(VOCAB_SIZE - 2):
    # len(vocab) 算出现有字典有多长。比如刚才里面有两个词，len 就是 2，新进来的词编号就是 2。
    vocab[word] = len(vocab)

# 把一段纯文本翻译成一串数字密码的加工厂函数（把新闻转换为词表编码列表）
def text_to_indices(text):
    tokens = simple_tokenize(text)
    # 🐍 语法小课堂：字典的 .get(单词, 默认值) 方法。如果单词在字典里，返回它的编号；如果不在，返回 '<UNK>' 的编号(1)
    indices = [vocab.get(w, vocab['<UNK>']) for w in tokens]
    
    if len(indices) < MAX_LEN:
        # 🐍 语法小课堂：在 Python 里，列表乘以一个数字，表示把列表复制多少遍。比如 [0] * 3 变成 [0, 0, 0]
        # 这里如果长度不够 128，就用 0 补齐。
        indices += [vocab['<PAD>']] * (MAX_LEN - len(indices))
    return indices # 返回长度固定为 128 的数字列表


# 🐍 语法小课堂：面向对象编程之 Class (类)。
# 类就像一张图纸。继承 (Dataset) 表示我们的图纸是在 Dataset 官方图纸基础上改的。
class NewsDataset(Dataset):
    # __init__ 是“构造函数”。当我们根据图纸造出一个实实在在的管理员对象时，第一步就会自动运行这里。
    # self 代表“自己”。self.data = data 就是把外面传进来的数据，存在自己肚子里。
    def __init__(self, data):
        self.data = data
        
    # __len__ 定义了别人问你“有多长”时你怎么回答
    def __len__(self):
        return len(self.data)
    
    # __getitem__ 定义了别人找你要“第 idx 个东西”时，你怎么拿给他
    def __getitem__(self, idx):
        text, label = self.data[idx] # 从自己肚子里的列表拿出对应的 (文字, 标签)
        
        # 调用上面的翻译函数，然后套一层 torch.tensor，把普通的 Python 列表变成 PyTorch 支持的“张量” (高级数组)
        text_tensor = torch.tensor(text_to_indices(text), dtype=torch.long)
        label_tensor = torch.tensor(label, dtype=torch.long)
        
        return text_tensor, label_tensor

# 实例化：按图纸造一个仓库管理员 dataset
dataset = NewsDataset(all_data)

# 雇佣搬运工 dataloader，告诉他去 dataset 仓库拿货，每次拿 32 篇 (batch_size)，打乱顺序拿 (shuffle)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

## 模块三：定义神经网络架构
**⚙️ 协同原理**：模块二搞定了数据怎么来，这里我们要定义数据**去哪里加工**。
我们要设计一个工厂（模型类 `TransformerClassifier`）。在初始化工厂 `__init__` 时，我们买好各种机器（词嵌入层、Transformer自注意力层、线性分类层）。
在 `forward` 函数里，我们铺设流水线履带：规定数据（一堆数字编号）先进第一台机器变成高维向量，再进第二台机器提取上下文语义，最后进分类机器打出各个类别的分数。

In [4]:
import math

# 继承神经网络老大哥 nn.Module
class TransformerClassifier(nn.Module):
    # 初始化函数：准备机器零件。参数是我们要告诉图纸的一些配置（比如词表多大，要几层等）
    def __init__(self, vocab_size, embed_dim, num_heads, num_layers, num_classes, dropout=0.1):
        super(TransformerClassifier, self).__init__() # 固定套路，先帮父类做初始化
        
        # 机器1：词嵌入 (把数字密码 变成 长长的小数向量)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        
        # 机器2：配置单层 Transformer
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,       # 输入向量的宽度
            nhead=num_heads,         # 注意力探照灯(多头)的个数
            dim_feedforward=embed_dim * 4, # 内部全连接层放大的倍数
            dropout=dropout,
            batch_first=True         # 声明我们的数据排布是 (批次, 句子长度, 词特征维度)
        )
        # 机器3：把刚才那一层叠 num_layers 次（比如叠2层）变成大编码器
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # 机器4：随机抛弃器 (防止过拟合的 Dropout)
        self.dropout = nn.Dropout(dropout)
        
        # 机器5：全连接线性分类器 (把前面提取的高深莫测的向量特征，强行拍碎映射到具体的5个分类上)
        self.classifier = nn.Linear(embed_dim, num_classes)
        
    # forward 是流水线，规定了从输入 x 开始，每台机器是怎么一步步起作用的
    def forward(self, x):
        # x 的长相：[32, 128] 代表 32 篇新闻，每篇 128 个数字
        
        # 第一步：查字典映射
        x = self.embedding(x) * math.sqrt(self.embedding.embedding_dim)
        # 现在 x 变胖了，长相：[32, 128, 128] (最后那个128是我们设置的 embed_dim 词向量维度)
        
        # 第二步：送进 Transformer，词与词之间开始互相“眉目传情”(自注意力计算)
        x = self.transformer_encoder(x)
        # x 长相没变：[32, 128, 128]，但里面的小数已经揉杂了上下文的语义
        
        # 第三步：降维池化。
        # 我们有 128 个词，怎么代表一篇文章？求个平均值吧。dim=1 意味着沿着“句子长度”那个维度压扁。
        x = x.mean(dim=1)
        # 现在 x 长相：[32, 128]。32篇文章，每篇文章浓缩成了一个 128维的终极特征。
        
        # 第四步：丢 Dropout
        x = self.dropout(x)
        
        # 第五步：得出分类打分
        logits = self.classifier(x)
        # 最终长相：[32, 5] (假设有5个类)。这代表这 32篇新闻在各个类别的得分(未经过概率转化)。
        
        return logits

# 获取真实新闻类别的个数
num_classes = len(labels_map)

# 根据图纸，把我们的加工厂(模型)正式实例化造出来！
model = TransformerClassifier(
    vocab_size=VOCAB_SIZE, 
    embed_dim=128, 
    num_heads=4, 
    num_layers=2, 
    num_classes=num_classes
)


## 模块四：让模型上考场（训练循环）
**⚙️ 协同原理**：这是所有模块协同大决战的地方！
我们要让 `dataloader` (搬运工) 不断地从 `dataset` 拿数据，喂给 `model` (加工厂) 算出 `outputs` (预测得分)。
然后我们要请一位考官 `criterion` (损失函数) 把得分和正确答案 `labels` 比对，算出差距 `loss`。
最后，我们要让 `optimizer` (优化器老师) 根据 `loss` 反向溯源，去扭转修改模型机器里的旋钮(参数)，让它下次不再犯同样的错。

In [5]:
# 🐍 语法小课堂：if ... else ... 
# 判断电脑里有没有 Nvidia GPU 加速卡 (cuda)。有就用，没有就老老实实用 CPU 算。
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(device)
model.to(device) # 把刚才造好的模型搬运到 GPU 显存里去

# 考官：多分类交叉熵损失函数。它专治分类错题。
criterion = nn.CrossEntropyLoss()

# 老师：Adam 优化器。
# model.parameters() 是一句指令，把工厂里所有能拧的旋钮（权重和偏置参数）都交给 Adam 老师控制。
# lr=1e-3 表示每次拧旋钮的幅度（学习率）。
optimizer = optim.Adam(model.parameters(), lr=1e-3)

epochs = 3 # 设定复习几遍题库

print("=== 开始激动人心的基础训练啦 ===")
for epoch in range(epochs): # 外层循环：第几遍刷题
    
    model.train() # 拨下工厂的“训练”开关。这会让 Dropout 开始随机断电，增加训练难度。
    
    total_loss = 0 
    correct = 0    
    total = 0      
    
    # 内层循环：每次拉来 32 篇新闻和答案。
    # 🐍 语法小课堂：enumerate(dataloader) 会把批次编号赋给 batch_idx，把数据打包赋给后面。 
    # (inputs, labels) 是“解包”语法，因为 dataloader 返回的是元组，我们直接用两个变量接住它。
    for batch_idx, (inputs, labels) in enumerate(dataloader):
        inputs, labels = inputs.to(device), labels.to(device) # 数据也得送进 GPU
        
        # 🚀 【核心 5 步法，深度学习永恒的铁律】 🚀
        
        # 1. 老师说：清空上次的错题记录！
        optimizer.zero_grad()
        
        # 2. 学生做题：前向传播，通过前文写的流水线，得到分数矩阵。
        outputs = model(inputs)
        
        # 3. 考官判卷：算算预测分数和标准答案的差距 (损失 Loss)。
        loss = criterion(outputs, labels)
        
        # 4. 反思找原因：反向传播！PyTorch 的魔法机制，自动根据微积分链式法则，算出每个旋钮该怎么扭才能让 Loss 变小。
        loss.backward()
        
        # 5. 执行修改：老师动手拧旋钮！参数被更新，模型变得更聪明了一点点。
        optimizer.step()
        
        # ---- 下面是汇报进度的代码，不影响训练 ----
        total_loss += loss.item() 
        
        # outputs.max(1) 沿着类别的那个维度找最大值。
        # 返回的 predicted 是网络认为概率最高的类别数字。
        _, predicted = outputs.max(1)
        total += labels.size(0) 
        
        # 看预测对不对。对了就加 1。
        correct += predicted.eq(labels).sum().item()
        
        # 每隔 100 车，汇报一次当下的成绩
        if (batch_idx + 1) % 100 == 0:
            acc = 100. * correct / total
            print(f"大循环(Epoch) [{epoch+1}/{epochs}], 小批次 [{batch_idx+1}], 错题大小(Loss): {loss.item():.4f}, 准确率(Acc): {acc:.2f}%")

print("🎉 基础版模型训练完成！跑通这个过程，你已经跨入深度学习的大门了！")

cuda
=== 开始激动人心的基础训练啦 ===
大循环(Epoch) [1/3], 小批次 [100], 错题大小(Loss): 0.7271, 准确率(Acc): 55.88%
大循环(Epoch) [1/3], 小批次 [200], 错题大小(Loss): 0.3557, 准确率(Acc): 69.69%
大循环(Epoch) [1/3], 小批次 [300], 错题大小(Loss): 0.4846, 准确率(Acc): 75.73%
大循环(Epoch) [2/3], 小批次 [100], 错题大小(Loss): 0.2846, 准确率(Acc): 92.41%
大循环(Epoch) [2/3], 小批次 [200], 错题大小(Loss): 0.3199, 准确率(Acc): 92.19%
大循环(Epoch) [2/3], 小批次 [300], 错题大小(Loss): 0.2406, 准确率(Acc): 92.41%
大循环(Epoch) [3/3], 小批次 [100], 错题大小(Loss): 0.1889, 准确率(Acc): 94.69%
大循环(Epoch) [3/3], 小批次 [200], 错题大小(Loss): 0.1641, 准确率(Acc): 94.22%
大循环(Epoch) [3/3], 小批次 [300], 错题大小(Loss): 0.2017, 准确率(Acc): 94.22%
🎉 基础版模型训练完成！跑通这个过程，你已经跨入深度学习的大门了！


## 模块五：深度解析 —— 刚才的训练过程到底发生了什么？

你可能看到屏幕上打印出了 `Loss` 越来越小，`Acc` (准确率) 越来越高，但代码背后到底发生了怎样的“化学反应”？我们来细细拆解：

1. **关于初始状态的“混沌”**：
   当我们刚实例化 `model = TransformerClassifier(...)` 时，模型里面所有的权重（那一堆 `w` 和 `b`）都是**完全随机的数字**。就像一个刚出生的婴儿，根本不懂什么是“科技”，什么是“体育”。
   这时候输入一篇科技新闻，模型在全连接层 `classifier(x)` 输出的 5 个类别的得分可能是 `[0.1, 0.2, 0.15, -0.3, 0.05]`，完全是在瞎蒙。

2. **关于 Loss（损失函数）的压迫感**：
   `criterion(outputs, labels)` 这句代码是整个学习的驱动力。如果正确答案是 0 类（科技），但模型给 0 类打的分很低，给 1 类打的分很高，CrossEntropyLoss 就会算出一个非常巨大的数字（比如 Loss = 5.8）。这个巨大的数字代表着“惩罚”。

3. **关于 Backward（反向传播）的奇迹**：
   当执行 `loss.backward()` 时，PyTorch 在后台使用微积分（链式法则），从输出层一路往回算：“如果我要把这个 5.8 的 Loss 降下来，全连接层的权重该怎么调？Transformer 的注意力权重该怎么调？甚至最底层的词嵌入矩阵该怎么调？”。它给每一个参数都计算出了一个调整方向（**梯度**）。

4. **关于 Step（梯度下降）的进化**：
   `optimizer.step()` 就是真正执行修改的一步。Adam 优化器根据刚才算出的梯度，微微旋转了模型里几百万个旋钮。于是，当下一批极其类似的新闻再送进来时，由于旋钮已经被优化过了，模型预测正确的概率就变大了，Loss 就变小了。

## 模块六：残酷的现实 —— 基础版 Transformer 的致命局限性

恭喜你，你已经用上面的代码成功训练了一个微型 Transformer。看起来一切都很完美对吧？

**但是，准备迎接学术界和工业界的毒打吧。**

如果你试图用这套代码去训练一个真正的“大”模型（比如把层数 `num_layers` 从 2 层变成 50 层，词维度 `embed_dim` 从 128 变成 1024），你会惊恐地发现：**模型彻底瘫痪了！**

在大型工程中，上面这个“最简基础版”会暴露出极其致命的局限性：

### 💣 局限性一：深层网络的“梯度消失”与“梯度爆炸” (The Scaling Curse)
在我们的代码里，数据 $x$ 要穿过 `num_layers` 层 Transformer。如果只有 2 层，误差反向传播时很容易传到底层。
但如果是 50 层呢？误差在反向传播时，每经过一层都要做一次矩阵乘法。如果每次乘的数都小于 1，乘了 50 次后，传到底层的误差就变成了 0（**梯度消失**，底层参数彻底罢工死机）；如果每次乘的数都大于 1，乘 50 次后数字就大到超出了电脑的计算极限，变成 `NaN`（**梯度爆炸**，模型直接崩溃）。

### 💣 局限性二：LayerNorm 的位置陷阱 (Post-LN vs Pre-LN)
在 PyTorch 默认的 `nn.TransformerEncoderLayer` 中，它使用的是一种叫做 **Post-LN (后置层归一化)** 的结构。也就是说，数据在做完自注意力计算后，才进行归一化。
学术界后来发现，**Post-LN 在深层网络中极度不稳定**！在深层 Post-LN 架构中，靠近输出层的参数梯度会异常巨大，而靠近输入层的参数梯度微乎其微。这会导致优化器 `Adam` 在刚开始训练时，瞬间把顶层参数更新崩塌。
为了救命，以前的人们只能用一种叫 **Learning Rate Warmup（学习率预热）** 的黑魔法，也就是一开始极度限制学习率，慢慢再放大。非常难以调参。

### 💣 局限性三：初始化的蝴蝶效应
在这份基础代码中，所有的参数（词嵌入矩阵、全连接层）都是用 PyTorch 默认的随机分布初始化的。
在小模型里，这点随机波动无所谓。但在几百层的巨型 Transformer 里，随着残差连接（$x = x + f(x)$）的一层层累加，前向传播到达顶层时，数值的“方差”（波动范围）会被放大约几百倍。模型一出生的输出就是一个极其狂暴、巨大波动的状态，Loss 直接算不出来。

---

## 🚀 引出破局之道：向 DeepNet 与进阶架构进发

难道 Transformer 注定只能是个几十层的“小矮子”吗？
当年微软和很多顶级 AI 实验室的研究员也面临着和你现在一样的绝望。他们为了把模型做到 1000 层，发表了大量重量级的论文，也就是你手头那**两份关于 DeepNet 和 Sub-LN/Post-LN 的 PDF 论文**所要解决的核心命题。

在那些论文中，大神们通过极其硬核的数学推导，提出了革命性的解决方案：
- **改变缩放因子**：引入 $\alpha$ 和 $\beta$ 缩放参数，压制残差连接带来的数值膨胀。
- **DeepNorm (深度归一化)**：重新设计归一化逻辑，完美解决了不需要 Warmup 也能让 1000 层网络平稳训练的世界级难题。
- **精妙的初始化机制**：不再随便随机初始化，而是根据层数 $N$ 动态缩放初始权重。

**准备好了吗？** 我们已经踏平了基础理论的平原，接下来，我们将拿起那两份 PDF 论文作为武器，对现在这个基础版代码进行“外科手术式的改造”，攻克大模型训练的真正难关！